# SILVA — Ribosomal RNA Sequence Database

**SILVA** is a comprehensive database of aligned ribosomal RNA (rRNA) sequences for all three domains of life (Bacteria, Archaea, Eukarya). It provides high-quality, regularly updated reference databases for small (16S/18S, SSU) and large (23S/28S, LSU) subunit rRNA genes, used extensively in microbiome research and phylogenetics.

| Property | Value |
|---|---|
| URL | https://www.arb-silva.de |
| Current release | 138.2 |
| Sequences | Millions of rRNA sequences |
| Subunits | SSU (16S/18S) and LSU (23S/28S) |
| Used for | 16S amplicon analysis, taxonomic assignment, phylogenetics |

In [ ]:
import requests
import gzip
import io
from pathlib import Path

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to SILVA FTP server and confirm access
    * [x] Download SSU taxonomy tree file (`tax_slv_ssu_138.2.txt.gz`) with caching
    * [x] Download SSU taxon accession map (`taxmap_slv_ssu_ref_nr_138.2.txt.gz`) with caching
    * [x] Parse taxonomy and taxon-map files into Polars DataFrames with correct dtypes
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise taxonomy levels (domain/phylum/class/order/family/genus)
    * [ ] Count unique taxa per rank
    * [ ] Check for missing taxonomy paths
    * [ ] Map accessions to full lineage strings
* [ ] **Analysis**
    * [ ] Compute sequence count per phylum
    * [ ] Compare bacterial vs. archaeal vs. eukaryotic representation
    * [ ] Identify most species-rich genera
* [ ] **Visualization**
    * [ ] Sunburst chart of domain/phylum breakdown
    * [ ] Bar chart of sequences per domain
* [ ] **Statistical analysis**
    * [ ] Test for taxonomic rank distribution
    * [ ] Compare representation bias between domains

## 1. Ingest Data

### 1.1 SILVA FTP — Confirm Server Access

In [ ]:
SILVA_FTP_BASE = "https://ftp.arb-silva.de/current/Exports"
SILVA_RELEASE = "138.2"

# Probe a known small file to verify FTP-over-HTTPS connectivity
PROBE_URL = f"{SILVA_FTP_BASE}/taxonomy/tax_slv_ssu_{SILVA_RELEASE}.txt.gz"

resp = requests.head(PROBE_URL, timeout=30)
print(f"URL    : {PROBE_URL}")
print(f"Status : {resp.status_code} {resp.reason}")
print(f"Content-Length : {resp.headers.get('Content-Length', 'n/a')} bytes")
print(f"Content-Type   : {resp.headers.get('Content-Type', 'n/a')}")
print("\nFTP server reachable:", resp.status_code == 200)

### 1.2 Download SSU Taxonomy Tree

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# The SSU taxonomy tree maps each taxon path to a numeric taxid and taxonomic rank.
# Columns: path, taxid, rank, remark, release
# File size: ~1.5 MB compressed — small enough to cache locally.
TAX_SSU_URL = f"{SILVA_FTP_BASE}/taxonomy/tax_slv_ssu_{SILVA_RELEASE}.txt.gz"
TAX_SSU_PATH = DATA_DIR / f"tax_slv_ssu_{SILVA_RELEASE}.txt.gz"

if not TAX_SSU_PATH.exists():
    print(f"Downloading {TAX_SSU_PATH.name} ...")
    with requests.get(TAX_SSU_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(TAX_SSU_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 16):  # 64 KB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e3:.1f} / {total / 1e3:.1f} KB", end="\r")
    print(f"\nSaved to {TAX_SSU_PATH}")
else:
    print(f"Already cached: {TAX_SSU_PATH}  ({TAX_SSU_PATH.stat().st_size / 1e3:.1f} KB)")

### 1.3 Download SSU Taxon Accession Map

In [ ]:
# The taxon map links each sequence accession (with coordinate range) to its
# full taxonomy path and organism name.
# Columns: primaryAccession, start, stop, path, organism_name, taxid
# File size: ~50 MB compressed.
TAXMAP_SSU_URL = f"{SILVA_FTP_BASE}/taxonomy/taxmap_slv_ssu_ref_nr_{SILVA_RELEASE}.txt.gz"
TAXMAP_SSU_PATH = DATA_DIR / f"taxmap_slv_ssu_ref_nr_{SILVA_RELEASE}.txt.gz"

if not TAXMAP_SSU_PATH.exists():
    print(f"Downloading {TAXMAP_SSU_PATH.name} ...")
    with requests.get(TAXMAP_SSU_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(TAXMAP_SSU_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {TAXMAP_SSU_PATH}")
else:
    print(f"Already cached: {TAXMAP_SSU_PATH}  ({TAXMAP_SSU_PATH.stat().st_size / 1e6:.1f} MB)")

### 1.4 Parse Taxonomy Tree into Polars DataFrame

In [ ]:
# The taxonomy tree file is a tab-separated file with a header row.
# Polars reads gzip directly — no manual decompression needed.
#
# Expected columns:
#   path          : semicolon-delimited lineage string, e.g. "Bacteria;Firmicutes;..."
#   taxid         : integer SILVA taxon ID
#   rank          : taxonomic rank label (domain, phylum, class, order, family, genus, ...)
#   remark        : optional editorial note (often empty)
#   release       : SILVA release version where the entry was introduced

taxonomy = pl.read_csv(
    TAX_SSU_PATH,
    separator="\t",
    has_header=True,
)

print(f"Shape  : {taxonomy.shape}")
print(f"Schema : {taxonomy.schema}")
print()
print(taxonomy.head(10))
print()

# Distribution of taxonomic ranks
print("Unique ranks and their counts:")
print(
    taxonomy.group_by("rank")
    .len()
    .sort("len", descending=True)
)

### 1.5 Parse Taxon Accession Map into Polars DataFrame

In [ ]:
# The taxon map file links each sequence entry to its taxonomic path.
# Expected columns:
#   primaryAccession : INSDC accession number (e.g. AB000380)
#   start            : 1-based start coordinate of the rRNA region on the record
#   stop             : 1-based stop coordinate of the rRNA region on the record
#   path             : semicolon-delimited taxonomy path (same format as taxonomy tree)
#   organism_name    : free-text organism name as submitted to INSDC
#   taxid            : integer SILVA taxon ID linking to the taxonomy tree

taxmap = pl.read_csv(
    TAXMAP_SSU_PATH,
    separator="\t",
    has_header=True,
    schema_overrides={"taxid": pl.Int64},
)

print(f"Shape  : {taxmap.shape}")
print(f"Schema : {taxmap.schema}")
print()
print(taxmap.head(10))
print()
print(f"Unique accessions  : {taxmap['primaryAccession'].n_unique():,}")
print(f"Null taxid count   : {taxmap['taxid'].null_count():,}")